# Agente Mundial 2026
Flujo diario: busca partidos y estado de forma via Tavily, genera análisis, escribe TXT y envía email.

## 1. Instalación

In [26]:
%pip install -q langchain langchain-openai langgraph tavily-python requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración(Keys)

In [27]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"]       = "https://n8nprueba-resource.services.ai.azure.com"
os.environ["AZURE_OPENAI_API_KEY"]        = "Fhyf30hJicRBBXThBvTBLNfdNtxco39E3ld4ByG9h8VYM1RJCoMBJQQJ99CFACfhMk5XJ3w3AAAAACOGJMTZ"
os.environ["AZURE_OPENAI_DEPLOYMENT"]     = "gpt-4o-mini"      
os.environ["AZURE_OPENAI_API_VERSION"]    = "2024-02-15-preview"

os.environ["TAVILY_API_KEY"]              = "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"

os.environ["EMAIL_FROM"]                  = "alejandrobenitez91203@gmail.com"
os.environ["EMAIL_PASSWORD"]              = "wmms uhhl ynta djrn"
os.environ["EMAIL_TO"]                    = "alejandrobenitez91203@gmail.com"

## 3. Tools

In [28]:
import smtplib
from datetime import date
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from tavily import TavilyClient
from langchain.tools import tool


def _tavily_search(query: str, max_results: int = 5) -> str:
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    resp = client.search(query=query, max_results=max_results, search_depth="advanced")
    results = resp.get("results", [])
    if not results:
        return "Sin resultados."
    lines = []
    for r in results:
        lines.append(f"[{r['title']}]\n{r['content']}\nFuente: {r['url']}\n")
    return "\n".join(lines)


@tool
def get_matches_today() -> str:
    """
    Busca los partidos del Mundial 2026 programados para hoy.
    Devuelve equipos, hora y sede. Si no hay partidos hoy lo indica claramente.
    """
    today = date.today().strftime("%B %d %Y")
    query = f"FIFA World Cup 2026 matches today {today} schedule kickoff time"
    return _tavily_search(query)


@tool
def get_next_matches() -> str:
    """
    Busca los próximos partidos del Mundial 2026 cuando no hay partidos hoy.
    Devuelve los próximos encuentros ordenados por fecha, del más cercano al más lejano.
    """
    query = f"FIFA World Cup 2026 next matches schedule fixtures in chronological order starting from {date.today().strftime('%B %d %Y')}"
    return _tavily_search(query)


@tool
def get_team_form(team_name: str) -> str:
    """
    Busca el estado de forma reciente de una selección nacional.
    Incluye últimos resultados, jugadores destacados y momento actual del equipo.
    team_name: nombre del equipo en español o inglés (ej. 'España', 'Brazil', 'France').
    """
    query = f"{team_name} seleccion nacional forma reciente resultados Mundial 2026 jugadores clave"
    return _tavily_search(query, max_results=3)


@tool
def get_next_matches() -> str:
    """
    Busca todos los partidos del Mundial 2026 de los próximos 3 días,
    haciendo una búsqueda específica por cada fecha para no perder partidos.
    """
    from datetime import timedelta
    results = []
    for i in range(1, 4):
        day = date.today() + timedelta(days=i)
        day_str = day.strftime("%B %d %Y")
        day_iso = day.isoformat()
        query = f"FIFA World Cup 2026 all matches {day_str} complete schedule kickoff times"
        text = _tavily_search(query, max_results=8)
        results.append(f"=== Partidos del {day_iso} ===\n{text}")
    return "\n\n".join(results)


@tool
def send_email_with_file(filepath: str = "partidos.txt") -> str:
    """
    Envía el análisis por email: el texto va en el cuerpo del mensaje Y como adjunto TXT.
    El asunto se lee de email_subject.txt.
    """
    from_addr = os.environ["EMAIL_FROM"]
    to_addr   = os.environ["EMAIL_TO"]
    password  = os.environ["EMAIL_PASSWORD"].replace(" ", "")

    try:
        with open("email_subject.txt", "r", encoding="utf-8") as f:
            subject = f.read().strip()
    except FileNotFoundError:
        subject = f"Mundial 2026 — {date.today().isoformat()}"

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            body_text = f.read()
    except FileNotFoundError:
        return f"ERROR: no se encontró '{filepath}'. Llama primero a write_matches_txt."

    if not body_text.strip():
        return "ERROR: partidos.txt está vacío. Llama primero a write_matches_txt con el contenido."

    msg = MIMEMultipart()
    msg["From"]    = from_addr
    msg["To"]      = to_addr
    msg["Subject"] = subject
    msg.attach(MIMEText(body_text, "plain", "utf-8"))

    with open(filepath, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", f'attachment; filename="{filepath}"')
    msg.attach(part)

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=10) as server:
            server.login(from_addr, password)
            server.sendmail(from_addr, to_addr, msg.as_string())
        return f"Email enviado a {to_addr}. Asunto: '{subject}'."
    except smtplib.SMTPAuthenticationError:
        return "ERROR auth: verifica verificación en 2 pasos y contraseña de aplicación Gmail."
    except Exception as e:
        return f"ERROR al enviar email: {str(e)}"


## 4. Agente

In [29]:
from langchain_openai import AzureChatOpenAI
from langchain.agents import create_agent

tools = [get_matches_today, get_next_matches, get_team_form, write_matches_txt, send_email_with_file]

llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    temperature=0.7
)

system_prompt = """Eres un analista de fútbol especializado en apuestas deportivas, con el estilo narrativo de Paolo Maldini: elegante, directo, conocedor del juego. Trabajas para apostadores serios que necesitan información accionable, no titulares.

Sigue este flujo EXACTO sin saltarte ningún paso ni inventarte información:

PASO 1 — Llama a get_matches_today.

PASO 2a — Si hay partidos hoy:
  - Para cada partido: llama a get_team_form con el equipo local y con el visitante.
  - Construye el texto completo con el formato indicado abajo.
  - El asunto será: "Mundial 2026 - [Local] vs [Visitante] - [fecha hoy en formato DD/MM/YYYY]".
  - Llama a write_matches_txt pasando el texto completo y el asunto.
  - Llama a send_email_with_file.

PASO 2b — Si NO hay partidos hoy:
  - Llama a get_next_matches para obtener todos los partidos de los próximos 3 días.
  - IMPORTANTE: incluye ABSOLUTAMENTE TODOS los partidos encontrados de esos 3 días, sin omitir ninguno.
  - Ordénalos por fecha y hora, del más cercano al más lejano.
  - Para cada partido: llama a get_team_form con el equipo local y con el visitante.
  - Construye el texto completo con el formato indicado abajo.
  - El asunto será: "Mundial 2026 - Próximos partidos - [fecha hoy en formato DD/MM/YYYY]".
  - Llama a write_matches_txt pasando el texto completo y el asunto.
  - Llama a send_email_with_file.

FORMATO DEL TEXTO (usa exactamente esta estructura, sin markdown):

MUNDIAL 2026 — [fecha de hoy en formato DD/MM/YYYY]
================================================

[EQUIPO LOCAL] vs [EQUIPO VISITANTE]
Fecha: [DD/MM/YYYY a las HH:MM hora España (CEST, UTC+2)] | [sede / ciudad]

[2 líneas de contexto: qué se juegan, momento del torneo, grupo]

Claves para apostar:
- 1X2: [análisis de quién gana y por qué, con confianza alta/media/baja]
- Más/Menos 2.5 goles: [argumenta si el partido será abierto o cerrado]
- Ambos equipos marcan: [sí o no, razonado]
- Handicap: [si hay favorito claro, indica el handicap recomendado]
- Apuesta recomendada: [la opción con mejor valor esperado de todo el análisis, en negrita]

------------------------------------------------

[siguiente partido en el mismo formato...]
RECUERDA SIEMPRE — estos dos pasos son OBLIGATORIOS y no son opcionales:
1. Al terminar el análisis de TODOS los partidos, llama a write_matches_txt con el texto completo.
2. Inmediatamente después llama a send_email_with_file.
Si no ejecutas estos dos pasos el trabajo está incompleto.

IMPORTANTE:
- Las horas deben estar en hora española (CEST = UTC+2), no en UTC ni hora americana.
- Incluye TODOS los partidos encontrados, no filtres ni omitas ninguno.
- Basa el análisis en datos reales de forma, no en reputación histórica de los equipos.
- Escribe siempre el texto completo antes de llamar a write_matches_txt.
- Nunca llames a send_email_with_file sin haber llamado antes a write_matches_txt.
- Ordena los partidos de fecha más cercana a más lejana. El primer partido del texto debe ser el más próximo en el tiempo.
- get_next_matches ya busca día a día. Extrae TODOS los partidos mencionados en cada sección de fecha, no solo los primeros que aparezcan.
- Si un partido aparece mencionado en los resultados de búsqueda, inclúyelo. No filtres por relevancia."""

agent = create_agent(llm, tools, system_prompt=system_prompt)

## 5. Ejecutar

In [30]:
result = agent.invoke({"messages": [{"role": "user", "content": "Procesa los partidos del Mundial 2026 de hoy."}]})
print("\n--- Resultado final ---")
print(result["messages"][-1].content)


--- Resultado final ---
MUNDIAL 2026 — 11/06/2026

MÉXICO vs SUDÁFRICA
Fecha: 11/06/2026 a las 21:00 hora España (CEST, UTC+2) | Estadio Azteca, Ciudad de México

Ambos equipos buscan iniciar el torneo con un pie derecho. México, el anfitrión, tiene la presión de rendir ante su afición, mientras que Sudáfrica quiere dar la sorpresa en este debut.

Claves para apostar:
- 1X2: México es el favorito, con una confianza alta de que se lleve la victoria en casa.
- Más/Menos 2.5 goles: El partido podría ser abierto, con más de 2.5 goles, dado el ataque de México y las debilidades defensivas de Sudáfrica.
- Ambos equipos marcan: Sí, ambos equipos tienen capacidad ofensiva.
- Handicap: Un -1 a favor de México parece adecuado.
- Apuesta recomendada: **México gana y más de 2.5 goles.**

------------------------------------------------

KOREA REPUBLIC vs CZECHIA
Fecha: 11/06/2026 a las 04:00 hora España (CEST, UTC+2) | Estadio Akron, Guadalajara, México

Korea Republic y Czechia debutan en un gru